In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


# BSM L06F — Face enrollment: 5 users, camera capture, local model

To nie jest lab z Pythonem. Pracujesz głównie na projekcie Android Studio / Kotlin.
Notebook służy jako formularz odpowiedzi i opis przepływu treningu backbone w Colab.

Student ma:
- skonfigurować 5 wierszy użytkowników
- dodać popup edycji zdjęć
- opisać aparat jako główne wejście i galerię jako backup
- przygotować backbone w Colab i eksport TFLite
- pokazać, gdzie model trafi w Android Studio


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    answers[task_id] = str(final_answer)
    print(f"Zapisano answers[{task_id}] ({len(str(final_answer))} znaków)")
    print(final_answer)
    # wysyłka do backendu zadania
    return final_answer


## 1. Install and imports
W tym kroku przygotowujesz środowisko notebooka. Jeśli używasz Colab, uruchom komórkę z instalacją bibliotek. Jeśli używasz lokalnego Jupyter, zainstaluj te same pakiety lokalnie.


In [ ]:
!pip -q install tensorflow-datasets
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

INPUT_SHAPE = (96, 96, 3)
EMBEDDING_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 12


## 2. Download dataset
Pobierasz publiczny zbiór twarzy z etykietami. W tej wersji używamy `lfw` jako prostego przykładu dla małego backbone.


In [ ]:
dataset_name = 'lfw'
builder = tfds.builder(dataset_name)
builder.download_and_prepare()
train_ds_raw = tfds.load(dataset_name, split='train[:80%]', as_supervised=True)
val_ds_raw = tfds.load(dataset_name, split='train[80%:]', as_supervised=True)


## 3. Preprocess
Obrazy mają zostać przeskalowane do `96x96`, znormalizowane do zakresu `[0, 1]` i zbatchowane. Android musi używać tego samego kontraktu wejścia.


In [ ]:
def collect_class_names(dataset):
    names = set()
    for _, label in tfds.as_numpy(dataset):
        if isinstance(label, bytes):
            names.add(label.decode('utf-8'))
        else:
            names.add(str(label))
    return sorted(names)

class_names = collect_class_names(train_ds_raw)
num_classes = len(class_names)
label_lookup = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(class_names),
        values=tf.constant(list(range(num_classes)), dtype=tf.int64),
    ),
    default_value=-1,
)

def preprocess(image, label):
    if isinstance(image, dict):
        image = image.get('image', image)
    if image.dtype == tf.string:
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image = tf.cond(
        tf.equal(tf.rank(image), 2),
        lambda: tf.expand_dims(image, -1),
        lambda: image,
    )
    image = tf.cond(
        tf.logical_and(tf.equal(tf.rank(image), 3), tf.equal(tf.shape(image)[-1], 1)),
        lambda: tf.image.grayscale_to_rgb(image),
        lambda: image,
    )
    image = tf.image.resize(image, INPUT_SHAPE[:2])
    image = tf.cast(image, tf.float32) / 255.0
    label = label_lookup.lookup(label)
    return image, label

train_ds = train_ds_raw.map(preprocess).shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


## 4. Model architecture
W tej części budujesz mały model CNN w `tensorflow.keras`. Używamy warstw `Conv2D`, `MaxPooling2D`, `GlobalAveragePooling2D` i końcowej warstwy `Dense`, żeby zamienić obraz 96x96x3 na embedding o rozmiarze 32.

Ten backbone jest wspólną bazą dla wszystkich użytkowników. W Colabie uczysz go na publicznym zbiorze twarzy, a potem eksportujesz do TFLite. Na Androidzie ten sam kontrakt wejścia musi pozostać taki sam, żeby model dało się uruchomić bez zmiany kodu aplikacji.

W tej sekcji pracujesz głównie z bibliotekami: `tensorflow`, `tensorflow_datasets`, `keras` i `layers`.


In [ ]:
def build_backbone(input_shape=INPUT_SHAPE, embedding_size=EMBEDDING_SIZE):
    inputs = keras.Input(shape=input_shape, name='face_input')
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embedding_size, activation='relu', name='embedding')(x)
    return keras.Model(inputs, x, name='tiny_face_backbone')

def build_classifier(num_classes):
    inputs = keras.Input(shape=INPUT_SHAPE, name='face_input')
    backbone = build_backbone()
    x = backbone(inputs)
    outputs = layers.Dense(num_classes, activation='softmax', name='identity_head')(x)
    return keras.Model(inputs, outputs, name='tiny_face_classifier')

model = build_classifier(num_classes)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 4b. Validation before training
Zanim uruchomisz trening, sprawdź czy dane i model mają zgodny kontrakt. Ta komórka pobiera jedną próbkę z `train_ds`, pokazuje jej kształt oraz liczbę klas. Jeśli tu jest błąd, naprawiasz go zanim rozpoczniesz `model.fit(...)`.


In [ ]:
sample_batch = next(iter(train_ds))
sample_images, sample_labels = sample_batch
print('sample_images.shape =', sample_images.shape)
print('sample_labels.shape =', sample_labels.shape)
print('num_classes =', num_classes)
assert sample_images.shape[1:] == INPUT_SHAPE
assert sample_images.dtype.is_floating


## 5. Train
W tej komórce uruchamiasz trening `model.fit(...)` na `train_ds` i sprawdzasz wyniki na `val_ds`. To jest moment, w którym model uczy się rozpoznawać klasy z przygotowanego zbioru danych.

Po treningu zapisujesz wynik walidacji do `val_mse` i `val_accuracy`. Te wartości są potem używane w `F01` i `F02`, więc student nie wpisuje ich ręcznie.


In [ ]:
import matplotlib.pyplot as plt

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

history_data = history.history
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history_data.get('loss', []), label='train loss')
axes[0].plot(history_data.get('val_loss', []), label='val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history_data.get('accuracy', []), label='train acc')
axes[1].plot(history_data.get('val_accuracy', []), label='val acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

plt.tight_layout()
plt.show()


## Evaluation
Ta komórka liczy metryki walidacyjne. `F01` i `F02` są brane automatycznie z `val_mse` i `val_accuracy`.


In [ ]:
eval_loss, eval_accuracy = model.evaluate(val_ds, verbose=0)
val_mse = float(eval_loss)
val_accuracy = float(eval_accuracy)
print(f'val_mse={val_mse:.4f}')
print(f'val_accuracy={val_accuracy:.4f}')


## 6. Export backbone
Po treningu eksportujesz sam backbone do pliku TFLite. Używasz `tf.lite.TFLiteConverter`, żeby zamienić model Keras na plik, który można wczytać w Androidzie.

W tej wersji eksportujesz tylko część bazową modelu, a nie pełną głowę klasyfikacyjną. Android ma wykonać tylko lokalne dopasowanie i późniejsze wnioskowanie.


In [ ]:
backbone_only = keras.Model(model.input, model.get_layer('tiny_face_backbone').output, name='tiny_face_backbone_export')
converter = tf.lite.TFLiteConverter.from_keras_model(backbone_only)
tflite_model = converter.convert()
with open('tiny_face_backbone.tflite', 'wb') as f:
    f.write(tflite_model)
with open('tiny_face_labels.txt', 'w') as f:
    f.write('\n'.join(class_names))


## 7. Download finished model
Po eksporcie uruchamiasz komórkę pobierania. W Colabie pojawi się automatyczne pobranie plików `tiny_face_backbone.tflite` i `tiny_face_labels.txt`. W Jupyterze dostajesz linki do plików.

To jest ostatni krok po stronie notebooka: student bierze gotowy model i kopiuje go do projektu Android Studio.


In [ ]:
try:
    from google.colab import files
    files.download('tiny_face_backbone.tflite')
    files.download('tiny_face_labels.txt')
except Exception:
    from IPython.display import FileLink, display
    display(FileLink('tiny_face_backbone.tflite'))
    display(FileLink('tiny_face_labels.txt'))


## 8. Where to put the files
Kopiujesz oba pliki do `student/apps/lesson_f_app/app/src/main/assets/`. Aplikacja Android ładuje je z tych samych ścieżek, więc nazwy plików muszą być dokładnie takie same jak w notebooku.

W praktyce oznacza to, że notebook kończy się eksportem, a Android Studio zaczyna pracę od zaimportowania gotowych artefaktów modelu.


## 9. Android runtime codes
Aplikacja pokazuje kody ukończenia w zależności od stanu sesji, runnera i testów.


## F01 — Training MSE
Po uruchomieniu komórki evaluation `final_answer` ma zostać wyliczony automatycznie z `val_mse` i zapisany z 2 miejscami po przecinku.


In [ ]:
# auto from evaluation
final_answer = f"{val_mse:.2f}"


In [ ]:
zapisz_i_wyslij("F01", final_answer)


## F02 — Training accuracy
Po uruchomieniu komórki evaluation `final_answer` ma zostać wyliczony automatycznie z `val_accuracy` i zapisany z 2 miejscami po przecinku.


In [ ]:
# auto from evaluation
final_answer = f"{val_accuracy:.2f}"


In [ ]:
zapisz_i_wyslij("F02", final_answer)


## F03 — Signed-in code
Po uruchomieniu aplikacji i wejściu na ekran signed in wpisz 4-cyfrowy kod pokazany przez app. Kod ma pochodzić z działającego stanu aplikacji.


In [ ]:
# kod stanu signed-in generowany przez aplikację
signed_in_code = "7421"
final_answer = signed_in_code


## Evaluation
Run the model on validation data. F01 and F02 are computed automatically from the evaluation output, so you do not type them by hand.


In [ ]:
eval_loss, eval_accuracy = model.evaluate(val_ds, verbose=0)
val_mse = float(eval_loss)
val_accuracy = float(eval_accuracy)
print(f'val_mse={val_mse:.4f}')
print(f'val_accuracy={val_accuracy:.4f}')


In [ ]:
zapisz_i_wyslij("F03", final_answer)


## F04 — Runner ready code
Po przejściu testów runtime wpisz 4-cyfrowy kod pokazany, gdy runner jest gotowy do inference. Ten kod ma pochodzić ze stanu aplikacji, nie z opisu zadania.


In [ ]:
# kod stanu runner-ready generowany przez aplikację
runner_code = "6184"
final_answer = runner_code


In [ ]:
zapisz_i_wyslij("F04", final_answer)


## F05 — Tests pass code
Po przejściu testów w Android Studio wpisz 4-cyfrowy kod potwierdzający, że wszystkie testy przeszły. To ma być wynik testów albo stanu aplikacji, a nie tekst wyjaśniający.


In [ ]:
# kod stanu tests-pass generowany przez aplikację
tests_pass_code = "9036"
final_answer = tests_pass_code


In [ ]:
zapisz_i_wyslij("F05", final_answer)
